# LAB | Hyperparameter Tuning

**Load the data**

Finally step in order to maximize the performance on your Spaceship Titanic model.

The data can be found here:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv

Metadata

https://github.com/data-bootcamp-v4/data/blob/main/spaceship_titanic.md

So far we've been training and evaluating models with default values for hyperparameters.

Today we will perform the same feature engineering as before, and then compare the best working models you got so far, but now fine tuning it's hyperparameters.

In [1]:
#Libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

In [2]:
spaceship = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv")
spaceship.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


Now perform the same as before:
- Feature Scaling
- Feature Selection


In [ ]:


from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler


features = spaceship.drop(columns="Transported")
target = spaceship["Transported"]


X_train, X_test, y_train, y_test = train_test_split(features, target, test_size=0.20, random_state=10)


X_train["Cabin"] = X_train["Cabin"].str[0]
X_test["Cabin"] = X_test["Cabin"].str[0]
X_train = X_train.drop(columns=["PassengerId", "Name"])
X_test = X_test.drop(columns=["PassengerId", "Name"])


num_cols = X_train.select_dtypes(include="number").columns
cat_cols = ["HomePlanet", "CryoSleep", "Cabin", "Destination", "VIP"]

num_imputer = SimpleImputer(strategy="mean")
num_imputer.fit(X_train[num_cols])                    
X_train[num_cols] = num_imputer.transform(X_train[num_cols])
X_test[num_cols] = num_imputer.transform(X_test[num_cols])

cat_imputer = SimpleImputer(strategy="most_frequent")
cat_imputer.fit(X_train[cat_cols])                      
X_train[cat_cols] = cat_imputer.transform(X_train[cat_cols])
X_test[cat_cols] = cat_imputer.transform(X_test[cat_cols])

encoder = OneHotEncoder(sparse_output=False, handle_unknown="ignore")
encoder.fit(X_train[cat_cols])                          

encoded_df_train = pd.DataFrame(encoder.transform(X_train[cat_cols]),
                                columns=encoder.get_feature_names_out(cat_cols),
                                index=X_train.index)
encoded_df_test = pd.DataFrame(encoder.transform(X_test[cat_cols]),
                               columns=encoder.get_feature_names_out(cat_cols),
                               index=X_test.index)


X_train_final = pd.concat([X_train.drop(columns=cat_cols), encoded_df_train], axis=1)
X_test_final = pd.concat([X_test.drop(columns=cat_cols), encoded_df_test], axis=1)

scaler = MinMaxScaler()
scaler.fit(X_train_final)
X_train_scaled = scaler.transform(X_train_final)
X_test_scaled = scaler.transform(X_test_final)

print("Missing values left in train:", X_train_final.isna().sum().sum())
print("Train shape:", X_train_scaled.shape, "| Test shape:", X_test_scaled.shape)

Missing values left in train: 0
Train shape: (6954, 24) | Test shape: (1739, 24)


- Now let's use the best model we got so far in order to see how it can improve when we fine tune it's hyperparameters.

In [4]:
#your code here
from sklearn.ensemble import RandomForestClassifier

randomforest_clf = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)
randomforest_clf.fit(X_train_scaled, y_train)
pred = randomforest_clf.predict(X_test_scaled)

- Evaluate your model

In [5]:
#your code here
from sklearn.metrics import accuracy_score, classification_report

print("Accuracy (before tuning):", accuracy_score(y_test, pred))
print(classification_report(y_test, pred))

Accuracy (before tuning): 0.8016101207590569
              precision    recall  f1-score   support

       False       0.81      0.79      0.80       883
        True       0.79      0.81      0.80       856

    accuracy                           0.80      1739
   macro avg       0.80      0.80      0.80      1739
weighted avg       0.80      0.80      0.80      1739



**Grid/Random Search**

For this lab we will use Grid Search.

- Define hyperparameters to fine tune.

In [6]:
#your code here
param_grid = {
    "n_estimators": [100, 200],          # number of trees
    "max_depth": [5, 10, 15, None],      # depth of each tree (None = no limit)
    "min_samples_leaf": [1, 3, 5],       # minimum number of rows in each leaf
}

- Run Grid Search

In [7]:
from sklearn.model_selection import GridSearchCV

grid_search = GridSearchCV(RandomForestClassifier(random_state=42),
                           param_grid,
                           cv=5,                 # 5-fold cross-validation on the TRAIN set
                           scoring="accuracy",   # classes are balanced (~50/50), so accuracy is fine
                           n_jobs=-1)            # use all CPU cores

grid_search.fit(X_train_scaled, y_train)

print("Best parameters:", grid_search.best_params_)
print("Best cross-validation accuracy:", round(grid_search.best_score_, 4))

Best parameters: {'max_depth': 15, 'min_samples_leaf': 5, 'n_estimators': 100}
Best cross-validation accuracy: 0.8014


- Evaluate your model

In [8]:
best_model = grid_search.best_estimator_
pred_tuned = best_model.predict(X_test_scaled)

print("Accuracy (after tuning):", accuracy_score(y_test, pred_tuned))
print(classification_report(y_test, pred_tuned))

Accuracy (after tuning): 0.8050603795284647
              precision    recall  f1-score   support

       False       0.82      0.79      0.80       883
        True       0.79      0.82      0.81       856

    accuracy                           0.81      1739
   macro avg       0.81      0.81      0.81      1739
weighted avg       0.81      0.81      0.81      1739

